## Data Preparation
-----

This notebook focuses on generating **nested training datasets** based on total audio duration.

We create progressively larger subsets (e.g., 1h → 2h → 5h → 10h), where each dataset includes all samples from the previous one. This supports consistent and reproducible training experiments.

- Uses a single shuffled manifest  
- Builds subsets using cumulative audio duration  
- Outputs separate CSV manifests for each duration  

> **Note:** Audio files are **not copied**. All subsets reference the same underlying audio paths, making the process fast and storage-efficient.


## 1. Python Setup

In [10]:
## Enviroment helper variable to help 
# when running the notebook in different environments (e.g. local vs colab)
ENV = "local"  # options: "local", "colab"

In [11]:
import sys
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path
from functools import partial

import pandas as pd
from datasets import DatasetDict, Dataset, Audio, Features, Value
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
)

# Add project root (recommended)
if ENV == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")   
else:
    PROJECT_SRC = Path().cwd().parents[0]

# Add project src to path to allow imports from src folder
sys.path.append(str(PROJECT_SRC))

# Import functions from src.train.train_whisper
from src.train.train_whisper import (
    load_config,
    DataCollatorSpeechSeq2SeqWithPadding,
    prepare_whisper_batch,
    compute_asr_metrics,
    build_training_args,
    evaluate_holdout_set,
)

from src.data.data_utils import load_audio_data  


## 2. Input Folder and Other Configuration

### 2.1 Input Folders

In [12]:
# Base data directory
DIR_BASE = Path.cwd().parents[0]
DIR_DATA = DIR_BASE.joinpath('data')

# Direcotory for test data and manifest file
DIR_TEST = DIR_DATA / "test"
FILE_MANIFEST_TEST = DIR_TEST / "metadata.csv"

# Directory for dev data and manifest file
DIR_DEV = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"

# Directory for nested duration based data
DIR_DEV_NESTED_DURATION = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE/"configs/whisper_params.yaml"



### 2.2 Hugging Face Hub Log in

In [13]:
# Log into Hugging Face Hub (optional, required if you want to push the model to the hub)
if ENV == "colab":
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 2.3 Device for Training

In [14]:
# Check if CUDA is available and set the device accordingly
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cpu


## 2. Load Training Data

In [15]:
# ===================================
# LOAD TRAIN AND TEST DATASETS 
# ===================================
# Lets use one hour training data for quick testing
file_manifest_1h = DIR_DEV_NESTED_DURATION / "train_1h.csv"
dataset_train = load_audio_data(file_manifest_1h, audio_dir=DIR_DEV)
dataset_test = load_audio_data(FILE_MANIFEST_TEST, audio_dir=DIR_TEST,audio_fname_col="audio_filename", 
                               split_data=False, duration_col="duration_seconds")

Total duration : 1.00 hrs  (278 utterances)
  train       :   244 utterances  |  0.90 hrs  (89.9%)
  validation  :    34 utterances  |  0.10 hrs  (10.1%)
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


## 3. Train Whisper Model 

### 3.1 Load Experiment Configuration 

In [16]:
# ======================================
# 1. LOAD CONFIG
# ======================================
config = load_config(FILE_CONFIG)
config

{'experiment': {'name': 'whisper_large_v3_chichewa_base',
  'output_dir': 'outputs/whisper_large_v3_chichewa_base',
  'hub_model_id': 'dmatekenya/whisper-large-v3-chichewa-base',
  'seed': 42},
 'model': {'model_name_or_path': 'openai/whisper-small',
  'language': 'swahili',
  'task': 'transcribe'},
 'training': {'per_device_train_batch_size': 2,
  'gradient_accumulation_steps': 1,
  'learning_rate': 1e-05,
  'warmup_steps': 5,
  'max_steps': 20,
  'gradient_checkpointing': False,
  'fp16': False,
  'predict_with_generate': True,
  'generation_max_length': 225,
  'save_strategy': 'steps',
  'save_steps': 10,
  'logging_steps': 5},
 'evaluation': {'eval_strategy': 'steps',
  'eval_steps': 10,
  'load_best_model_at_end': True,
  'metric_for_best_model': 'wer',
  'greater_is_better': False},
 'hub': {'push_to_hub': False, 'report_to': 'none'}}

### 3.2 Load Model and Processor

In [17]:
# ======================================
# 2. LOAD MODEL AND PROCESSOR
# ======================================

model_id = config["model"]["model_name_or_path"]
print(f"Loading model and processor from: {model_id}")

processor = WhisperProcessor.from_pretrained(
    model_id,
    language=config["model"]["language"],
    task=config["model"]["task"],
)

model = WhisperForConditionalGeneration.from_pretrained(model_id,torch_dtype=torch.float32)

model.generation_config.language = config["model"]["language"]
model.generation_config.task = config["model"]["task"]
model.generation_config.forced_decoder_ids = None

Loading model and processor from: openai/whisper-small


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 30333.85it/s]


### 3.3 Prepare Dataset for Whisper Training

In [18]:
# ================================
# 3. PREPARE DATASET
# ================================
# Prepare the training and test datasets for Whisper model training.
# This step applies the `prepare_whisper_batch` function to each batch in the datasets.
# It processes the audio and text columns, tokenizes the text, and prepares the input features for the model.
# The original columns are removed, keeping only the processed features required for training.

dataset_train = dataset_train.map(
    lambda batch: prepare_whisper_batch(
        batch,
        processor=processor,
        audio_column="audio",
        text_column="sentence",
    ),
    remove_columns=dataset_train["train"].column_names,
    num_proc=4,
)

# Keep audio_fname out of remove_columns so it survives into the dataset
cols_to_remove = [c for c in dataset_test.column_names if c != "audio_fname"]
dataset_test = dataset_test.map(
    lambda batch: {
        **prepare_whisper_batch(
            batch,
            processor=processor,
            audio_column="audio",
            text_column="sentence",
        ),
        "audio_fname": batch["audio_filename"],  # explicitly carry it through
    },
    remove_columns=cols_to_remove,
    num_proc=1,   # lambda can't be pickled for multiprocessing
)



Map (num_proc=1):   0%|          | 0/573 [00:00<?, ? examples/s]


KeyError: 'audio_filename'

### 3.4 Configure Trainer and Launch Training

In [ ]:
# ==========================================
# 4. DATA COLLATOR AND TRAINING ARGUMENTS
# ==========================================
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [ ]:
# ==========================================
# 5. DATA COLLATOR AND TRAINING ARGUMENTS
# ==========================================
# Build the training arguments for the Seq2SeqTrainer using 
# the provided configuration from the config object.
training_args = build_training_args(config)

In [ ]:
# ==========================================
# 6. SETUP TRAINER
# ==========================================
# Setup the Seq2SeqTrainer with the model, training arguments, 
# datasets, data collator, and metric computation function.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train["train"],
    eval_dataset=dataset_train["validation"],
    data_collator=data_collator,
    compute_metrics=partial(compute_asr_metrics, processor=processor),
    processing_class=processor.feature_extractor,
)

# 
trainer.train()

In [ ]:
dataset_test_sample = dataset_test.shuffle(seed=42).select(range(20))
df_results = evaluate_holdout_set(model, processor, dataset_test_sample, text_column="sentence", fname_column="audio_fname")

In [ ]:
dataset_test_sample